> **Ekip paylaşım kopyası:** Kayıtlı çıktılar ve widget durumu dosya boyutunu azaltmak için temizlenmiştir; analiz hücreleri korunur. Veri dosyası Git deposuna dahil değildir. Kurulum, veri yolu ve mevcut veri dosyasını kopyalamadan kullanma seçenekleri için `../README.md` dosyasına bakın. `.venv` kernel'ini seçip Run All çalıştırın. Kaynak projedeki notebook ve çıktılar değiştirilmemiştir.

# DS02 veri yapısının keşfi

Bu notebook `data/N-CMAPSS_DS02-006.h5` dosyasını **salt okunur** açar. Model eğitimi, preprocessing veya dosyaya yazma yapılmaz. Büyük veri dizilerinin yalnızca ilk 10 satırı okunur; değişken adları küçük metadata dizilerinden alınır.

Gereken paketler: `h5py`, `numpy`, `pandas`. Jupyter üzerinden çalıştırmak için ayrıca `ipykernel` gerekir. Kaydedilmiş çıktılar projenin `.venv` ortamındaki gerçek Jupyter çekirdeğiyle üretilmiştir.


### Öğretici okuma rehberi ve değişken sözlüğü
[DS02 Türkçe değişken sözlüğü](../docs/DS02_Degisken_Sozlugu.md), 47 özgün değişkenin tam İngilizce/Türkçe adlarını, birimlerini, bileşenlerini ve yorum sınırlarını açıklar. A kimlik/etiket, W koşul, X_s fiziksel ölçülebilir yanıt, X_v sanal model çıktısı, T simülasyon iç sağlığı, Y/RUL hedeftir. Fiziksel sensör adı bütün yanıtların gerçek bir arızalı motordan kaydedildiği anlamına gelmez; N-CMAPSS motor yanıtları simüle edilmiştir.

Makale Tables 4–8'e göre sıcaklık °R, basınç psia, hız rpm, kütlesel debi pps/lbm/s'dir. TRA birimi %; stall marjları kaynakta boyutsuz gösterilir. Grafik kodu değiştirilmediği için “özgün değer” etiketlerinin fiziksel birimleri sözlükten okunmalıdır. Önceki birim-doğrulama notlarının üzerine bu yeni kaynak açıklaması eklenmiştir; hesaplamalar değişmemiştir.


In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from IPython.display import display

# Notebook proje kökünden veya notebooks klasöründen çalıştırılabilir.
# Paylaşım kopyası: veri yolu bilgisayardan bağımsız çözülür.
import os

def resolve_ncmapss_paths(start=None):
    working_directory = Path.cwd().resolve() if start is None else Path(start).resolve()
    project_root = None
    for parent in [working_directory, *working_directory.parents]:
        if parent.name == "n_cmapss" and (parent / "notebooks").is_dir():
            project_root = parent
            break
        candidate = parent / "n_cmapss"
        if (candidate / "notebooks").is_dir():
            project_root = candidate
            break
    if project_root is None:
        raise FileNotFoundError("Notebook'u ortak repo, n_cmapss veya n_cmapss/notebooks klasöründen çalıştırın.")
    external_path = os.environ.get("N_CMAPSS_H5")
    h5_path = Path(external_path).expanduser().resolve() if external_path else project_root / "data" / "N-CMAPSS_DS02-006.h5"
    return project_root, h5_path

project_root, h5_path = resolve_ncmapss_paths()
if not h5_path.is_file():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {h5_path}. HDF5 repoya dahil değildir. "
        "Dosyayı yerel n_cmapss/data/ altında sağlayın veya kernel ortamında "
        "N_CMAPSS_H5 değişkenini mevcut dosyanın tam yoluna ayarlayın. README'ye bakın."
    )

print(f"Dosya: {h5_path}")
print(f"Boyut: {h5_path.stat().st_size:,} bayt")


Dosya: /Users/ilkinsoydas/Desktop/projeler/vscode/predictive-maintance/n_cmapss/data/N-CMAPSS_DS02-006.h5
Boyut: 2,450,472,504 bayt


## 1. Bütün HDF5 nesneleri

HDF5 içinde grup ve dataset farklı nesnelerdir. Aşağıdaki kod bütün alt nesneleri dolaşır; datasetlerin `shape` ve `dtype` özelliklerini okur, içeriklerini belleğe yüklemez. Grupların boyut ve veri tipi alanları uygulanabilir değildir. Her erişimde `with` bloğu dosyayı işlem sonunda kapatır.


### Grup, satır ve sütun farklı kavramlardır
Bir HDF5 datasetinin `(satır, sütun)` boyutu fiziksel cihaz sayısı değildir: X_s_dev'nin her satırında 14 kanal vardır. Altı dev dizisi aynı satır pozisyonlarıyla hizalanır. 17 kök nesne ile birleşik 47 değişken farklı sayılardır. [Grup açıklamaları](../docs/DS02_Degisken_Sozlugu.md#a-yardimci).

In [2]:
inventory = []
with h5py.File(h5_path, "r") as h5:
    def inspect(name, obj):
        inventory.append({
            "isim": name,
            "nesne_türü": "Dataset" if isinstance(obj, h5py.Dataset) else "Grup",
            "boyut": str(obj.shape) if isinstance(obj, h5py.Dataset) else "—",
            "veri_tipi": str(obj.dtype) if isinstance(obj, h5py.Dataset) else "—",
        })
    h5.visititems(inspect)
inventory_df = pd.DataFrame(inventory)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(inventory_df)


,isim,nesne_türü,boyut,veri_tipi
0,A_dev,Dataset,"(5263447, 4)",float64
1,A_test,Dataset,"(1253743, 4)",float64
2,A_var,Dataset,"(4,)",|S5
3,T_dev,Dataset,"(5263447, 10)",float64
4,T_test,Dataset,"(1253743, 10)",float64
5,T_var,Dataset,"(10,)",|S12
6,W_dev,Dataset,"(5263447, 4)",float64
7,W_test,Dataset,"(1253743, 4)",float64
8,W_var,Dataset,"(4,)",|S4
9,X_s_dev,Dataset,"(5263447, 14)",float64


## 2. Değişken isimleri

Küçük `*_var` dizileri okunur. Bayt dizileri UTF-8 metne çevrilir; her isim kendi sütun indeksiyle gösterilir. Büyük ölçüm dizileri bu aşamada okunmaz.


### Makale ile metadata nasıl eşleştirilir?
X_s isimleri dosyada T24 ile başlar; makale Table 5 Wf/Nf/Nc ile başlar. Eşleştirmeyi sıra numarasıyla değil isimle yapın. Y_var yoktur; Y_dev'e RUL adı notebook'ta verilir. phi'nin ölçek ve LPT_flow_mod'un basılı açıklama belirsizliği [sözlükte](../docs/DS02_Degisken_Sozlugu.md#birimler) ayrıca belirtilir.

In [3]:
variable_keys = ["A_var", "W_var", "X_s_var", "X_v_var", "T_var"]
variable_names = {}
with h5py.File(h5_path, "r") as h5:
    for key in variable_keys:
        raw = np.asarray(h5[key][()]).reshape(-1)
        variable_names[key] = [
            value.decode("utf-8").rstrip("\x00")
            if isinstance(value, (bytes, np.bytes_)) else str(value)
            for value in raw
        ]
        print(f"\n{key} ({len(variable_names[key])} değişken)")
        display(pd.DataFrame({"sütun_indeksi": range(len(variable_names[key])),
                              "değişken": variable_names[key]}))



A_var (4 değişken)


,sütun_indeksi,değişken
0,0,unit
1,1,cycle
2,2,Fc
3,3,hs



W_var (4 değişken)


,sütun_indeksi,değişken
0,0,alt
1,1,Mach
2,2,TRA
3,3,T2



X_s_var (14 değişken)


,sütun_indeksi,değişken
0,0,T24
1,1,T30
2,2,T48
3,3,T50
4,4,P15
5,5,P2
6,6,P21
7,7,P24
8,8,Ps30
9,9,P40



X_v_var (14 değişken)


,sütun_indeksi,değişken
0,0,T40
1,1,P30
2,2,P45
3,3,W21
4,4,W22
5,5,W25
6,6,W31
7,7,W32
8,8,W48
9,9,W50



T_var (10 değişken)


,sütun_indeksi,değişken
0,0,fan_eff_mod
1,1,fan_flow_mod
2,2,LPC_eff_mod
3,3,LPC_flow_mod
4,4,HPC_eff_mod
5,5,HPC_flow_mod
6,6,HPT_eff_mod
7,7,HPT_flow_mod
8,8,LPT_eff_mod
9,9,LPT_flow_mod


## 3. Satır sayısı kontrolü ve ilk 10 satır

`A_dev` yardımcı bilgiler (unit, cycle vb.), `W_dev` uçuş koşulları, `X_s_dev` sensör ölçümleri, `X_v_dev` sanal sensörler, `T_dev` bozulma parametreleri ve `Y_dev` RUL değerlerini içerir. Birleştirme öncesinde altı datasetin satır sayıları ve sütun metadatası doğrulanır.

Yalnızca `[:10]` dilimleri alınır. Tablolar aynı orijinal satır indeksiyle yatay birleştirilir; unit veya cycle üzerinden eşleştirme yapılmaz. Bu işlem veri dönüşümü veya preprocessing değildir.


### İlk 10 satırı danışmana açıklama
unit motor, cycle uçuş, Fc=3 uzun uçuş sınıfıdır. hs=1 normal, hs=0 anormal bozulma dönemi; 0 anlık motor duruşu değildir. RUL kalan uçuş sayısıdır. Aynı uçuşta RUL sabitken sensörler koşullarla değişebilir. [hs–HI–RUL–EOL ayrımı](../docs/DS02_Degisken_Sozlugu.md#saglik-kavramlari).

In [4]:
dev_keys = ["A_dev", "W_dev", "X_s_dev", "X_v_dev", "T_dev", "Y_dev"]
metadata_map = dict(zip(dev_keys[:-1], variable_keys))
with h5py.File(h5_path, "r") as h5:
    row_counts = {key: h5[key].shape[0] for key in dev_keys}
    display(pd.DataFrame.from_dict(row_counts, orient="index", columns=["satır_sayısı"]))
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"dev satır sayıları eşit değil: {row_counts}")
    print(f"Kontrol başarılı: bütün dev datasetlerinde {next(iter(row_counts.values())):,} satır var.")
    sample_count = min(10, next(iter(row_counts.values())))
    frames = []
    for key in dev_keys:
        dataset = h5[key]
        column_count = dataset.shape[1] if dataset.ndim == 2 else 1
        columns = ["RUL"] if key == "Y_dev" else variable_names[metadata_map[key]]
        if dataset.ndim not in (1, 2) or len(columns) != column_count:
            raise ValueError(f"{key}: boyut ve sütun isimleri uyumsuz")
        values = dataset[:sample_count]
        frames.append(pd.DataFrame(values.reshape(sample_count, column_count),
                                   columns=columns, index=pd.RangeIndex(sample_count, name="satır_indeksi")))
    preview = pd.concat(frames, axis=1, verify_integrity=True)
    if not {"unit", "cycle", "RUL"}.issubset(preview.columns):
        raise ValueError("Beklenen unit, cycle veya RUL sütunu eksik")
print(f"Birleşik tablo: {preview.shape[0]} satır, {preview.shape[1]} sütun")
with pd.option_context("display.max_columns", None, "display.width", 240):
    display(preview)


,satır_sayısı
A_dev,5263447
W_dev,5263447
X_s_dev,5263447
X_v_dev,5263447
T_dev,5263447
Y_dev,5263447


Kontrol başarılı: bütün dev datasetlerinde 5,263,447 satır var.
Birleşik tablo: 10 satır, 47 sütun


,unit,cycle,Fc,hs,alt,Mach,TRA,T2,T24,T30,T48,T50,P15,P2,P21,P24,Ps30,P40,P50,Nf,Nc,Wf,T40,P30,P45,W21,W22,W25,W31,W32,W48,W50,SmFan,SmLPC,SmHPC,phi,fan_eff_mod,fan_flow_mod,LPC_eff_mod,LPC_flow_mod,HPC_eff_mod,HPC_flow_mod,HPT_eff_mod,HPT_flow_mod,LPT_eff_mod,LPT_flow_mod,RUL
satır_indeksi,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,2.0,1.0,3.0,1.0,10005.0,0.448497,76.903748,502.420918,600.148034,1438.498187,1818.027714,1228.129848,15.806267,11.577097,16.046971,20.126624,331.293679,336.631827,12.629361,2160.926416,8591.373490,3.855337,2797.727106,352.494060,62.768751,2420.557738,228.486927,228.487065,26.498785,15.899271,215.844851,228.411666,16.648833,9.898130,25.376144,41.893990,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
1,2.0,1.0,3.0,1.0,10013.0,0.447741,76.903748,502.326114,600.055894,1438.350208,1817.682618,1227.879113,15.795477,11.568235,16.036017,20.113218,331.109867,336.446748,12.623033,2160.909333,8590.972866,3.852319,2797.209328,352.300260,62.734717,2418.935113,228.382424,228.383505,26.486552,15.891931,215.745634,228.307014,16.639222,9.904927,25.380549,41.884434,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
2,2.0,1.0,3.0,1.0,10017.0,0.448938,77.079529,502.416067,600.210756,1439.109101,1820.020627,1229.422522,15.807747,11.574866,16.048474,20.130956,331.753181,337.082502,12.637951,2161.819062,8593.031745,3.866201,2800.736255,352.965971,62.851755,2421.245669,228.651138,228.661083,26.519340,15.911604,216.019054,228.592279,16.649823,9.923503,25.318848,41.953848,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
3,2.0,1.0,3.0,1.0,10024.0,0.449883,77.079529,502.469893,600.369717,1439.240230,1819.188327,1228.538726,15.816360,11.578198,16.057218,20.146716,331.819136,337.162828,12.631509,2162.768666,8593.781545,3.863328,2799.352383,353.050081,62.873081,2422.941615,228.786949,228.768625,26.532044,15.919226,216.121238,228.702994,16.653812,9.905518,25.361981,41.914342,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
4,2.0,1.0,3.0,1.0,10031.0,0.449379,77.079529,502.401271,600.298227,1439.064004,1818.963540,1228.389046,15.807513,11.571593,16.048236,20.135888,331.626003,336.966936,12.624872,2162.619544,8593.220200,3.860818,2799.034785,352.844959,62.836600,2421.608933,228.651685,228.653631,26.518460,15.911076,216.008509,228.584788,16.649031,9.897465,25.363994,41.911503,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
5,2.0,1.0,3.0,1.0,10040.0,0.447993,77.079529,502.248778,600.132240,1438.787598,1818.564681,1228.168979,15.790096,11.557944,16.030555,20.113280,331.319000,336.655384,12.616795,2162.391423,8592.379660,3.856593,2798.465003,352.518727,62.778178,2418.790474,228.466770,228.469611,26.496723,15.898034,215.830871,228.397252,16.630032,9.909184,25.364223,41.904433,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
6,2.0,1.0,3.0,1.0,10044.0,0.448749,77.079529,502.297384,600.153022,1438.802338,1818.881160,1228.441326,15.792667,11.561284,16.033164,20.115416,331.339635,336.672635,12.618247,2162.172722,8592.301697,3.858025,2798.972822,352.536790,62.779772,2419.196996,228.460392,228.461991,26.495823,15.897494,215.823301,228.388645,16.639463,9.905753,25.354068,41.917385,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
7,2.0,1.0,3.0,1.0,10048.0,0.449820,77.079529,502.376490,600.241423,1438.946992,1818.997364,1228.446281,15.800091,11.567046,16.040702,20.125383,331.464924,336.800620,12.619357,2162.366948,8592.770144,3.859438,2799.122088,352.670807,62.804120,2420.596850,228.544635,228.539220,26.504945,15.902967,215.899000,228.468251,16.654342,9.901054,25.356844,41.916889,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74
8,2.0,1.0,3.0,1.0,10056.0,0.448938,77.079529,502.269929,600.148341,1438.776202,1818.528459,1228.084356,15.788539,11.557252,16.028974,20.111582,331.255580,336.591137,12.611525,2162.413322,8592.335539,3.855745,2798.409850,352.451452,62.766222,2418.874006,228.429525,228.428633,26.491882,15.895129,215.791703,228.355596,16.642967,9.904451,25.366197,41.903239,0.0,0.0,0.0,0.0,0.0,0.0,-0.000638,0.0,0.0,0.0,74


## Kapsam ve bellek kullanımı

Ölçüm verilerinin tamamı okunmadı. Belleğe yalnızca beş değişken adı dizisi ve altı dev datasetinin ilk 10 satırı alındı. HDF5 dosyası bütün erişimlerde `r` modunda açıldı ve kapatıldı. Eğitim, ölçekleme, filtreleme veya başka preprocessing uygulanmadı.
